# 📊 Fase 1: EDA y Preparación de Datos
## Sistema de Alerta Temprana MINSAL

**Proyecto**: Sistema de Alerta Temprana y Predicción de Demanda en Urgencias Respiratorias
**Autor**: Ignacio Fernández (Nafery) - ML Lead
**Datos**: 2024-2025 | 16 regiones | 658 establecimientos

---

## FASE 1: Business Understanding

**Objetivo del negocio**: Pasa de gestión reactiva a gestión preventiva de crisis respiratorias invernales, con alerta temprana a **96 horas** (4 días).

**Objetivo de ML**: Predecir la demanda diaria en urgencias respiratorias a nivel establecimiento, integrando features climáticas, de contaminación y demográficas, con lags biológicos (TempMin_Lag_4D, MP2_5_Lag_2D).

---

## FASE 2: Data Understanding

**Datasets (4 fuentes)**:
1. **Salud (DEIS/MINSAL)**: Atenciones respiratorias por establecimiento, fecha, rango etario, causa (CIE-10 J00-J99)
2. **Temperatura (SENAMHI/CR2)**: Mínima y máxima diaria por región
3. **Precipitaciones (Open-Meteo)**: Agua caída diaria en mm por región
4. **Contaminación (SINCA/Open-Meteo)**: MP2.5 y MP10 en μg/m³ por región

---

## FASE 3: Data Preparation

Este notebook prepara los datos para el modelo. Ejecute primero este notebook, luego el de modelado.

## Pipeline de Integración — Visión General

El pipeline consta de 2 scripts modulares:

```
data/raw/ ──────────────────────────────► src/pipelines/unificar_datos.py ──► data/interim/df_unificado.parquet
   ├── AtencionesUrgencia2024.csv         (integra salud + clima + contaminación)
   ├── AtencionesUrgencia2025.csv         (a nivel establecimiento + fecha)
   ├── Datos Por Región/Temperatura*.csv  (clima SENAMHI)
   ├── Contaminacion 2024 y 2025.csv      (Open-Meteo PM2.5/PM10)
   └── Precipitaciones 2024 y 2025.csv    (Open-Meteo precipitación)
                                              │
                                              ▼
                                    data/interim/df_unificado.parquet
                                              │
                                              ▼
                                  src/pipelines/feature_engineering.py
                                          (lags, flags, discretización)
                                              │
                                              ▼
                              ✅ data/processed/df_modelo.parquet (dataset final)
```

In [ ]:
# Importaciones
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

BASE_DIR = Path('/Users/naferyh/Desktop/Proyectos/Proyecto de título')

## Cargar Datos Unificados

Si el dataset no existe todavía, ejecuta primero los scripts de integración:

In [ ]:
# Verificar si el dataset unificado existe
df_path = BASE_DIR / 'data/interim/df_unificado.parquet'
if not df_path.exists():
    print('⚠️  Dataset no encontrado. Ejecutando pipeline de integración...')
    import subprocess, sys
    subprocess.run([sys.executable, str(BASE_DIR / 'src/pipelines/unificar_datos.py')], check=True)

print('Cargando dataset unificado...')
df = pd.read_parquet(df_path)
print(f'✅ Dataset cargado: {len(df):,} filas x {len(df.columns)} columnas')

### 1. Análisis de dimensiones y perfil de datos

In [ ]:
# Análisis de dimensiones
print(f"Total de filas: {len(df):,}")
print(f"Columnas: {list(df.columns)}")

# Rango de fechas
df['Fecha'] = pd.to_datetime(df['Fecha'])
print(f"\nRango de fechas: {df['Fecha'].min()} → {df['Fecha'].max()}")

# Dimensiones clave
print(f"\nRegiones: {df['Region'].nunique()}")
print(f"Establecimientos: {df['IdEstablecimiento'].nunique()}")
print(f"Tipos de establecimiento:\n{df['TipoEstablecimiento'].value_counts().head()}")

### 2. Valores nulos y datos faltantes

In [ ]:
# Análisis de nulos
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
null_df = pd.DataFrame({'Valores nulos': df.isnull().sum(), '%': null_pct})
null_df = null_df[null_df['Valores nulos'] > 0].sort_values('%', ascending=False)
print("Columnas con valores nulos:")
display(null_df)

# Visualización
plt.figure(figsize=(12, 6))
cols_with_nulls = null_df.index.tolist()
if cols_with_nulls:
    sns.barplot(x=null_df['%'].values, y=null_df.index, palette='Reds_r')
    plt.xlabel('% Valores Nulos')
    plt.title('Distribución de Valores Nulos por Columna')
    plt.tight_layout()
    plt.show()

### 3. Estadísticas descriptivas — Variables numéricas

In [ ]:
# Estadísticas descriptivas
numeric_cols = ['Total', 'Menores_1', 'De_1_a_4', 'De_5_a_14', 'De_15_a_64', 'De_65_y_mas', 'TempMin', 'TempMax', 'Precipitacion_mm', 'MP2_5', 'MP10']
available_numeric = [c for c in numeric_cols if c in df.columns]

print("📊 Estadísticas descriptivas:")
display(df[available_numeric].describe().T)

# Distribución de atenciones por rango etario
age_cols = ['Menores_1', 'De_1_a_4', 'De_5_a_14', 'De_15_a_64', 'De_65_y_mas']
age_totals = df[age_cols].sum()
plt.figure(figsize=(10, 5))
sns.barplot(x=age_totals.index, y=age_totals.values, palette='viridis')
plt.xticks(rotation=30)
plt.ylabel('Total de atenciones (2024-2025)')
plt.title('Distribución de Atenciones por Rango Etario')
plt.tight_layout()
plt.show()

### 4. Distribución geográfica

In [ ]:
# Atenciones por región
region_totals = df.groupby('Region')['Total'].sum().sort_values(ascending=False)

plt.figure(figsize=(14, 6))
sns.barplot(x=region_totals.values, y=region_totals.index, palette='coolwarm')
plt.xlabel('Total de Atenciones Respiratorias (2024-2025)')
plt.title('Demanda Respiratoria por Región')
plt.tight_layout()
plt.show()

print("Top 5 regiones con mayor demanda:")
print(region_totals.head())

### 5. Patrones temporales

In [ ]:
# Tendencia temporal
df['Mes'] = df['Fecha'].dt.month
df['Año'] = df['Fecha'].dt.year

monthly = df.groupby(['Año', 'Mes'])['Total'].sum().reset_index()
monthly['Fecha_mes'] = pd.to_datetime(dict(year=monthly['Año'], month=monthly['Mes'], day=1))

plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly, x='Fecha_mes', y='Total', marker='o')
plt.title('Tendencia Mensual de Atenciones Respiratorias (2024-2025)')
plt.ylabel('Total de atenciones')
plt.tight_layout()
plt.show()

# Estacionalidad por mes
plt.figure(figsize=(10, 5))
monthly_avg = df.groupby('Mes')['Total'].mean()
sns.barplot(x=monthly_avg.index, y=monthly_avg.values, palette='YlOrRd')
plt.xlabel('Mes')
plt.ylabel('Promedio de atenciones')
plt.title('Patrón Estacional por Mes')
plt.tight_layout()
plt.show()

### 6. Relación clima-contaminación-atenciones

In [ ]:
# Correlación entre variables climáticas y atenciones
climate_cols = ['Total', 'TempMin', 'TempMax', 'Precipitacion_mm', 'MP2_5', 'MP10']
available_climate = [c for c in climate_cols if c in df.columns and df[c].notna().sum() > 0]

corr = df[available_climate].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.3f')
plt.title('Matriz de Correlación: Atenciones vs Variables Climáticas')
plt.tight_layout()
plt.show()

# Relación temperatura-atenciones
plt.figure(figsize=(10, 5))
plt.scatter(df['TempMin'], df['Total'], alpha=0.3, s=1)
plt.xlabel('Temperatura Mínima (°C)')
plt.ylabel('Total Atenciones')
plt.title('Relación Temperatura Mínima vs Atenciones Respiratorias')
plt.tight_layout()
plt.show()

## Generar Dataset Final para Modelado

Una vez completado el EDA, aplicamos feature engineering y guardamos el dataset procesado.
El script `feature_engineering.py` genera lags, flags y estadísticos regionales.

In [ ]:
# Aplicar feature engineering
fe_path = BASE_DIR / 'src/pipelines/feature_engineering.py'
processed_path = BASE_DIR / 'data/processed/df_modelo.parquet'

if not processed_path.exists():
    print('🔄 Aplicando feature engineering...')
    import subprocess, sys
    subprocess.run([sys.executable, str(fe_path)], check=True)

# Cargar dataset final
df_modelo = pd.read_parquet(processed_path)
print(f"\n✅ Dataset listo para modelado: {len(df_modelo):,} filas x {len(df_modelo.columns)} columnas")
print(f"Columnas: {list(df_modelo.columns)}")

## ✅ Resumen de Preparación

| Paso | Estado | Output |
|------|--------|--------|
| Integración 3 fuentes | ✅ | `data/interim/df_unificado.parquet` |
| Feature engineering (lags/flags) | ✅ | Aplicado en pipeline |
| Dataset final | ✅ | `data/processed/df_modelo.parquet` |

**Próximo paso**: Abrir `notebooks/02_modelado_predictivo.ipynb` con el dataset procesado.